# MCP Introduction with LangChain

##  Objective

An AI agent needs **tools** to do real work, like searching documentation, reading files or calling an API. **MCP (Model Context Protocol)** is a standard way to plug those tools into an agent. You could think of it as **"USB-C for AI"**: any MCP server works with any MCP-compatible agent.

In this notebook we will **connect a LangChain agent to a real MCP server** (the official LangChain documentation server), so the agent can look up live documentation and answer questions with it.

## 🗺️ Steps we will follow

| # | Step | What happens |
|---|------|--------------|
| 1 | **Setup** | Install packages, load the OpenAI API key, create the LLM |
| 2 | **Connect to an MCP server** | Point `MultiServerMCPClient` at `https://docs.langchain.com/mcp` |
| 3 | **Load the tools** | Ask the server which tools it offers and turn them into LangChain tools |
| 4 | **Build an agent** | Give the LLM those tools and ask it questions about LangChain |
| 5 | **Read resources** | Fetch read-only data (documents) that the server exposes |
| 6 | **Add a logging interceptor** | Watch every tool call the agent makes, as it happens |
| 7 | **Key takeaways** | Recap what we learned and where to go next |

> 💡 **Big idea to remember:** we never write the search tool ourselves. The MCP server provides it, and our agent simply *plugs in* and uses it.

## The Big Picture: Who is Who?

![MCP big picture: our notebook is the MCP client, the LangChain Documentation MCP Server provides 3 tools, 1 resource and 0 prompts](mcp_big_picture.svg)

### The 3 things an MCP server can offer

| Type | Simple meaning | Who uses it | This server offers |
|------|----------------|-------------|--------------------|
| 🔧 **Tools** | Actions ("do something") | The **AI agent** decides to call them | **3** (search, read pages, feedback) |
| 📄 **Resources** | Data ("read something") | **Our app** fetches it | **1** (LangChain guide) |
| 📝 **Prompts** | Ready-made prompt templates | The **user** picks one | **0** |

> 💡 **Remember:** the server is **built and hosted by LangChain**, but **our notebook is the client**. We could point the same client at any other MCP server (GitHub, Slack, a database...) by changing only the URL.

## Setup

In [1]:
%pip install -qU langchain langchain-openai langchain-mcp-adapters langgraph mcp python-dotenv "pydantic>=2,<3"


Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

# Load environment variables from .env file
load_dotenv()

# Verify API key is loaded
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY not found in environment variables. Please create a .env file with your API key.")

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

## LangChain Integration with MCP

### Using langchain-mcp-adapters

LangChain provides the `langchain-mcp-adapters` package, which offers a seamless way to connect LangChain applications to MCP servers. This package includes `MultiServerMCPClient`, which allows you to connect to multiple MCP servers and load their tools, resources, and prompts into LangChain agents.

### Key Benefits

- **Standardized Protocol**: Work with one protocol instead of juggling multiple APIs
- **Multiple Servers**: Connect to multiple MCP servers simultaneously
- **LangChain Native**: Tools, resources, and prompts are automatically LangChain-compatible
- **Stateless by Default**: Each tool call creates a fresh session, no context manager needed
- **Tool Interceptors**: Middleware pattern for customizing tool behavior

### The LangChain Documentation MCP Server

We'll use the LangChain documentation MCP server as our example:
- **MCP Server URL**: `https://docs.langchain.com/mcp`
- **Documentation Index**: `https://docs.langchain.com/llms.txt`
- **Transport**: HTTP (not stdio)
- **Provides**: Access to LangChain, LangGraph, and LangSmith documentation

This MCP server allows you to query the latest LangChain documentation programmatically, making it perfect for building agents that can answer questions about LangChain features and APIs.

### Connecting to the LangChain Documentation MCP Server

Now let's connect to the LangChain documentation MCP server using HTTP transport.

`MultiServerMCPClient` accepts a dictionary where keys are server names and values are configuration dicts specifying `transport` and connection details:

In [3]:
from langchain_mcp_adapters.client import MultiServerMCPClient

# Configure the LangChain documentation MCP server
# MultiServerMCPClient takes a dict of server configs
# Each config specifies transport type and connection details
mcp_client = MultiServerMCPClient({
    "langchain-docs": {
        "transport": "http",
        "url": "https://docs.langchain.com/mcp"
    }
})

print("MCP client configured for LangChain documentation server")
print(f"Server: langchain-docs → https://docs.langchain.com/mcp")

MCP client configured for LangChain documentation server
Server: langchain-docs → https://docs.langchain.com/mcp


### 📱 What did this cell actually do?

**Analogy:** we just **saved a contact in our phone**. We have **not called** anyone yet.

| In the code | In the phone analogy | Meaning |
|---|---|---|
| `"langchain-docs"` | The **contact name** | **Our own label.** It does not need to match anything on LangChain's side: `"docs"` or `"my-server"` would work too |
| `"transport": "http"` | **How** we reach them (call, WhatsApp, SMS...) | We talk to the server over the internet, because LangChain **hosts** it remotely |
| `"url"` | The **phone number** | Where the server lives. This is the one value that **must** be correct |

📞 The actual **call** happens in the next step, with `await mcp_client.get_tools()`. That is when the client connects, asks the server *"what tools do you have?"*, and hangs up again.

> 💡 **Try it:** change the URL to something wrong and re-run the cell above. It still prints *"configured"* without any error, because saving a wrong number in your phone works fine. You only find out it's wrong when you **call** it (`get_tools()`).

In [4]:
mcp_client

### Loading MCP Tools into LangChain

The `get_tools()` method retrieves all tools from connected MCP servers and converts them to LangChain-compatible tools.

**Note:** `MultiServerMCPClient` is stateless by default — each call to `get_tools()` internally creates and tears down fresh sessions automatically. No `async with` context manager is needed:

In [5]:
# Load tools from the MCP server
# The client is stateless: get_tools() creates ephemeral sessions under the hood
mcp_tools = await mcp_client.get_tools()

print(f"Loaded {len(mcp_tools)} tools from MCP server(s)")
print("\nAvailable tools:")
for tool in mcp_tools:
    print(f"  - {tool.name}: {tool.description[:80]}...")

Loaded 3 tools from MCP server(s)

Available tools:
  - search_docs_by_lang_chain: Search across the Docs by LangChain knowledge base to find relevant information,...
  - query_docs_filesystem_docs_by_lang_chain: Run a read-only shell-like query against a virtualized, in-memory filesystem roo...
  - submit_feedback: Report a problem with this documentation site so the docs team can fix it. Use w...


### 📞 What did `get_tools()` actually do?

**Continuing the phone analogy:** `get_tools()` is **a call** where we ask the server *"what services do you offer?"*. We **write the list down** and **hang up**. Every time we want one of those services later, **we call again**.

| Step | In the phone analogy | What really happens |
|---|---|---|
| 1 | Dial the number | Open a connection to `https://docs.langchain.com/mcp` |
| 2 | "Hello, who's this?" | **Handshake**: client and server agree on the protocol |
| 3 | "What services do you offer?" | The client asks for the list of tools |
| 4 | Write the list down | We receive 3 tools (name, description, inputs) and store them in `mcp_tools` |
| 5 | Hang up 📴 | The connection is **closed** |

So `mcp_tools` is **not a live connection**. It is our written-down list: each tool remembers *what it does* and *which number to call*. When the agent uses a tool later, the client **calls the server again**, runs the tool, and hangs up.

> 💡 **Remember:** `await` here means *"wait on the line until the server has given us the full list"*. The next line (`print(...)`) only runs **after** the list has arrived.

In [6]:
mcp_tools

[StructuredTool(name='search_docs_by_lang_chain', description='Search across the Docs by LangChain knowledge base to find relevant information, code examples, API references, and guides. Use this tool when you need to answer questions about Docs by LangChain, find specific documentation, understand how features work, or locate implementation details. The search returns contextual content with titles and direct links to the documentation pages. If you need the full content of a specific page, use the query_docs_filesystem tool to `head` or `cat` the page path (append `.mdx` to the path returned from search — e.g. a result at `/some/page` is read with `head -200 /some/page.mdx`).', args_schema={'type': 'object', 'properties': {'query': {'type': 'string', 'description': 'Search query'}}, 'required': ['query'], 'additionalProperties': False}, metadata={'title': 'Search documentation', 'readOnlyHint': True, 'destructiveHint': False, 'idempotentHint': True, 'openWorldHint': False}, handle_to

### Using MCP Tools with LangChain Agents

Now let's create a LangChain agent that can use these MCP tools to query the LangChain documentation.

We use `create_agent` from `langchain.agents`, with the `system_prompt` parameter to set the system message. We pass in the `mcp_tools` we already loaded, so there is no need to call the server again:

In [19]:
tools

[StructuredTool(name='search_docs_by_lang_chain', description='Search across the Docs by LangChain knowledge base to find relevant information, code examples, API references, and guides. Use this tool when you need to answer questions about Docs by LangChain, find specific documentation, understand how features work, or locate implementation details. The search returns contextual content with titles and direct links to the documentation pages. If you need the full content of a specific page, use the query_docs_filesystem tool to `head` or `cat` the page path (append `.mdx` to the path returned from search — e.g. a result at `/some/page` is read with `head -200 /some/page.mdx`).', args_schema={'type': 'object', 'properties': {'query': {'type': 'string', 'description': 'Search query'}}, 'required': ['query'], 'additionalProperties': False}, metadata={'title': 'Search documentation', 'readOnlyHint': True, 'destructiveHint': False, 'idempotentHint': True, 'openWorldHint': False}, handle_to

In [7]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

# Reuse the tools we already loaded (no need to call the server again)
tools = mcp_tools

# Create agent with the model and MCP tools
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="You are a helpful assistant that can answer questions about LangChain, LangGraph, and LangSmith by searching the official documentation. Always provide accurate, up-to-date information based on the documentation."
)

print(f"Agent created with {len(tools)} MCP tools")
print(f"Tools: {[tool.name for tool in tools]}")

Agent created with 3 MCP tools
Tools: ['search_docs_by_lang_chain', 'query_docs_filesystem_docs_by_lang_chain', 'submit_feedback']


### Querying LangChain Documentation

Let's test the agent by asking it questions about LangChain:

> 🔑 **Rule:** an agent that uses MCP tools is always called with **`await agent.ainvoke(...)`**, never with `agent.invoke(...)`.

In [8]:
# Query the agent about LangChain
question = "How do I create a LangChain agent with tools?"
print(f"Question: {question}\n")

result = await agent.ainvoke({
    "messages": [HumanMessage(content=question)]
})

print(f"Answer: {result['messages'][-1].content}")

Question: How do I create a LangChain agent with tools?

Answer: To create a LangChain agent with tools, you can follow these steps. Below are examples in both JavaScript and Python.

### JavaScript Example

1. **Install Required Packages**:
   ```bash
   npm install langchain zod @langchain/openai
   ```

2. **Create the Agent**:
   ```javascript
   import { createAgent, tool } from "langchain";
   import * as z from "zod";

   const getWeather = tool(
     (input) => `It's always sunny in ${input.city}!`,
     {
       name: "get_weather",
       description: "Get the weather for a given city",
       schema: z.object({
         city: z.string().describe("The city to get the weather for"),
       }),
     }
   );

   const agent = createAgent({
     model: "gpt-5.5",
     tools: [getWeather],
   });

   console.log(
     await agent.invoke({
       messages: [{ role: "user", content: "What's the weather in San Francisco?" }],
     })
   );
   ```

### Python Example

1. **Install Req

### Accessing MCP Resources

MCP resources provide read-only access to data. The `get_resources()` method returns LangChain `Blob` objects.

You can optionally filter by server name or specific URIs:

In [9]:
# Access MCP resources
# get_resources() is also stateless — no context manager needed
mcp_resources = await mcp_client.get_resources()


In [10]:
mcp_resources

[Blob 2597118890528]

In [11]:
print(mcp_resources[0].as_string())

---
name: langchain
description: Use when building AI agents, integrating language models with
  tools, orchestrating multi-step workflows, or deploying production agent
  systems. Reach for this skill when working with LangChain, LangGraph, Deep
  Agents, or LangSmith for agent development, testing, and deployment.
metadata:
  mintlify-proj: langchain
  version: "1.0"
---

# LangChain Skill

## Product summary

LangChain is an open-source framework for building AI agents and LLM applications. It provides three core layers: **LangChain** (high-level agent harness with `create_agent`), **LangGraph** (low-level orchestration runtime for stateful workflows), and **Deep Agents** (batteries-included harness with planning, filesystem, subagents, and memory). Use LangChain's `create_agent` for customizable agents, LangGraph for deterministic + agentic workflows, or Deep Agents for complex multi-step tasks. Deploy with LangSmith Cloud for managed infrastructure or self-host with LangSmith Depl

### Middleware and Tool Interceptors

One of the powerful features of `langchain-mcp-adapters` is the ability to use tool interceptors (middleware) to customize tool behavior. This allows you to:

- **Log tool calls**: Track when and how tools are used
- **Modify requests/responses**: Transform tool inputs or outputs
- **Add validation**: Ensure tool inputs meet certain criteria


Interceptors follow a handler callback pattern (similar to middleware in web frameworks). Any async callable matching the `(request, handler) -> result` signature works. Interceptors compose in an "onion" pattern where the first interceptor is the outermost layer.

Let's create a simple tool interceptor that logs tool usage:

In [12]:
from langchain_mcp_adapters.client import MultiServerMCPClient

# Tool interceptors are async callables with signature:
#   async def interceptor(request, handler) -> result
# - request: MCPToolCallRequest with .name and .args
# - handler: callable to invoke the next interceptor or the actual tool
# - return: MCPToolCallResult

async def logging_interceptor(request, handler):
    """Interceptor that logs tool calls and responses."""
    print(f"[LOG] Tool '{request.name}' called with input: {request.args}")
    result = await handler(request)
    print(f"[LOG] Tool '{request.name}' returned result: {str(result.content)[:100]}...")
    return result

# Create MCP client with tool interceptor
# Pass interceptors as a list via the tool_interceptors parameter
mcp_client_with_logging = MultiServerMCPClient(
    {
        "langchain-docs": {
            "transport": "http",
            "url": "https://docs.langchain.com/mcp"
        }
    },
    tool_interceptors=[logging_interceptor]
)

print("MCP client configured with logging interceptor")

MCP client configured with logging interceptor


### Complete Example: Agent with Logging

Let's create a complete example that uses the logging interceptor:

In [13]:
# Get tools from the client WITH the interceptor.
# The interceptor lives on the client, so these tools already carry it:
# do NOT pass tool_interceptors to create_agent (it has no such parameter).
tools = await mcp_client_with_logging.get_tools()

# Create agent
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="You are a helpful assistant that answers questions about LangChain documentation."
)

# Ask a question
question = "What is LangChain and what are its main features?"
print(f"\nQuestion: {question}\n")
print("=" * 60)

result = await agent.ainvoke({
    "messages": [HumanMessage(content=question)]
})

print("=" * 60)
print(f"\nFinal Answer:\n{result['messages'][-1].content}")


Question: What is LangChain and what are its main features?

[LOG] Tool 'search_docs_by_lang_chain' called with input: {'query': 'What is LangChain and its main features'}
[LOG] Tool 'search_docs_by_lang_chain' returned result: [TextContent(type='text', text='Title: Benefits\nLink: https://docs.langchain.com/oss/javascript/rel...

Final Answer:
LangChain is a framework designed for building applications that utilize large language models (LLMs). It provides a standardized interface and a set of tools that facilitate the development of LLM applications across various providers. Here are some of its main features:

1. **Provider Agnostic**: LangChain allows developers to access reasoning traces, citations, and built-in tools (like web search and code interpreters) using the same API, regardless of the underlying provider.

2. **Standard Model Interface**: It offers a unified interface for different types of models, such as chat models and embeddings, enabling easy switching between mode

### Key Takeaways

1. **langchain-mcp-adapters** provides a seamless way to integrate MCP servers with LangChain
2. **MultiServerMCPClient** accepts a dict of server configurations — no separate parameter classes needed
3. **Stateless by default** — each `get_tools()` or `get_resources()` call handles sessions internally, no `async with` needed
4. **HTTP Transport** is used for web-based MCP servers like the LangChain documentation server
5. **Tool Interceptors** are async callables with `(request, handler) -> result` signature, passed via `tool_interceptors=`
6. **`create_agent`** (from `langchain.agents`) takes the system message via `system_prompt=`. Interceptors go on the **client**, never on the agent

### Documentation References

- **LangChain Documentation MCP**: `https://docs.langchain.com/mcp`
- **Documentation Index**: `https://docs.langchain.com/llms.txt`
- **langchain-mcp-adapters**: [LangChain MCP Documentation](https://docs.langchain.com/oss/python/langchain/mcp)
- **API Reference**: [langchain-mcp-adapters Reference](https://reference.langchain.com/python/langchain_mcp_adapters/)

### Next Steps

- Explore other MCP servers from the [official MCP servers repository](https://github.com/modelcontextprotocol/servers)
- Build custom tool interceptors for your specific use cases
- Combine multiple MCP servers for comprehensive agent capabilities

In [14]:
from langchain_core.documents.base import Blob

blob = Blob.from_data("Hello, world!")

# Read the blob as a string
print(blob.as_string())

# Read the blob as bytes
print(blob.as_bytes())



Hello, world!
b'Hello, world!'


---

# 📘 Recap: Two Concepts Behind This Notebook

Two ideas appeared again and again in this notebook:

1. **`await`**: on every line that talks to the MCP server or the LLM
2. **Tool interceptors**: the "checkpoint" we used to log tool calls

Let's look at each one, step by step.

---

## 1️⃣ Synchronous vs Asynchronous

| | Synchronous (normal Python) | Asynchronous (`async` / `await`) |
|---|---|---|
| **How it works** | Finishes one task completely before starting the next | While one task is waiting, Python can work on another task |
| **While waiting** | Does nothing, just waits | Switches to another task that is ready |
| **Good for** | Quick calculations | Slow waiting: APIs, MCP servers, LLM calls |

☕ **Café analogy**
- **Synchronous:** the barista makes your coffee, and only *after* it's done takes the next order.
- **Asynchronous:** the barista starts your coffee, and *while the machine runs*, takes the next order.

**The two keywords:**
- `async def`: "this function contains some waiting"
- `await`: "pause *this* task here until the result is ready. Meanwhile, Python may run other tasks."

Let's measure the difference 👇 First, the **synchronous** way:

In [15]:
import time

def get_data_sync(source):
    print(f"Requesting {source}...")
    time.sleep(3)  # pretend we are waiting 3 seconds for an API
    print(f"Received data from {source}")

start = time.perf_counter()
get_data_sync("MCP server")
get_data_sync("OpenAI API")
print(f"\nSynchronous total time: {time.perf_counter() - start:.1f} seconds")

Requesting MCP server...
Received data from MCP server
Requesting OpenAI API...
Received data from OpenAI API

Synchronous total time: 6.0 seconds


Now the **same two requests**, the **asynchronous** way:

In [16]:
import asyncio
import time

async def get_data(source):
    print(f"Requesting {source}...")
    await asyncio.sleep(3)  # pretend we are waiting 3 seconds for an API
    print(f"Received data from {source}")

async def main():
    await asyncio.gather(  # start both tasks at the same time
        get_data("MCP server"),
        get_data("OpenAI API"),
    )

start = time.perf_counter()
await main()
print(f"\nAsynchronous total time: {time.perf_counter() - start:.1f} seconds")

Requesting MCP server...
Requesting OpenAI API...
Received data from MCP server
Received data from OpenAI API

Asynchronous total time: 3.0 seconds


### 🔍 What happened?

| | Total time | Why |
|---|---|---|
| **Synchronous** | ~6 seconds | 3 s + 3 s: the second request only starts after the first one finishes |
| **Asynchronous** | ~3 seconds | Both requests wait **at the same time** |

Look at the **print order** in the async output: both `Requesting...` lines appear first, then both `Received...` lines. That shows Python switched between the two tasks while they were waiting.

> ✅ **Key takeaway:** `await` lets Python switch to another ready task while the current task is waiting.

💡 **Notebook vs. `.py` file:** in a notebook you can write `await` directly in a cell. In a normal Python script, put your code in an `async def main()` and run it with `asyncio.run(main())`.

---

## 2️⃣ Tool Interceptors

**In one sentence:** an interceptor is a **checkpoint between the agent and the tool**. Every tool request passes through it on the way out, and every result passes through it on the way back.

![Interceptor flow: the agent's request passes through the interceptor (inside the MCP client, on our computer) on its way to the tool on the MCP server, and the result passes through it again on the way back](interceptor_flow.svg)

🏢 **Analogy: the receptionist.** A visitor (the agent) can't walk straight into the manager's office (the tool on the server). Everyone passes the receptionist first, both on the way **in** and on the way **out**.

> 📍 **Where exactly?** On **our computer**, inside the MCP client: `MultiServerMCPClient(..., tool_interceptors=[...])`. Not inside the agent, and not on the server.

### Every interceptor has the same shape

```python
async def my_interceptor(request, handler):
    # BEFORE: look at, check or change the request
    result = await handler(request)   # let the visitor in: the real tool runs on the server
    # AFTER: look at or change the result
    return result
```

| Part | Meaning |
|---|---|
| `request` | What the agent wants: `request.name` (which tool) and `request.args` (the inputs) |
| `handler` | The door to the manager's office. **If you don't call it, the tool never runs.** |

### What can an interceptor do?

| Job | The receptionist... | Example |
|---|---|---|
| 📝 **Log** | writes every visitor in the visitor book | Print or save every tool call and its result, e.g. for an audit trail. **We try this one below.** |
| ✅ **Validate** | turns a visitor away if the request is not acceptable | Reject a search query that is too short, before it reaches the server |
| ✏️ **Change input / output** | rewrites a message on the way in or on the way out | Add *"in Python"* to every search, or shorten a very long answer |

> 💡 All three jobs use the **same shape** shown above. Only the code in the BEFORE and AFTER parts changes.


### 🛠️ Example: logging every tool call

We'll build **one** interceptor, `log_calls`, that writes down every tool call: **before** the tool runs (which tool, which inputs) and **after** it answers.

💰 We call the tool **directly** here (no LLM), so this example costs nothing.


In [17]:
from langchain_mcp_adapters.client import MultiServerMCPClient

# LOG: write down every tool call
async def log_calls(request, handler):
    print(f"[LOG] BEFORE: calling '{request.name}' with {request.args}")
    result = await handler(request)   # the real tool runs on the server
    print(f"[LOG] AFTER:  '{request.name}' answered")
    return result

# Plug the interceptor into the CLIENT (not into the agent)
mcp_client_with_log = MultiServerMCPClient(
    {
        "langchain-docs": {
            "transport": "http",
            "url": "https://docs.langchain.com/mcp"
        }
    },
    tool_interceptors=[log_calls],
)

logged_tools = {tool.name: tool for tool in await mcp_client_with_log.get_tools()}
print("Client ready with the log interceptor")


Client ready with the log interceptor


In [18]:
search = logged_tools["search_docs_by_lang_chain"]

result = await search.ainvoke({"query": "What is MCP?"})
print("\nResult (first 300 characters):", result[0]["text"][:300], "...")


[LOG] BEFORE: calling 'search_docs_by_lang_chain' with {'query': 'What is MCP?'}
[LOG] AFTER:  'search_docs_by_lang_chain' answered

Result (first 300 characters): Title: Relay elicitation requests
Link: https://docs.langchain.com/langsmith/server-mcp-elicitation#relay-elicitation-requests
Page: langsmith/server-mcp-elicitation
Content: ## Relay elicitation requests

An interrupt value whose `type` is `mcp_elicitation` carries MCP elicitation requests. Agent S ...


### 🔍 What happened?

| Line you see | When | Meaning |
|---|---|---|
| `[LOG] BEFORE: calling 'search_docs_by_lang_chain' with {'query': 'What is MCP?'}` | Before the tool runs | The receptionist wrote down **who** came and **what** they asked |
| `[LOG] AFTER: ... answered` | After the server answered | The receptionist saw the answer go back |
| `Result: ...` | At the end | The tool's answer reached us **unchanged**: logging only watches, it doesn't change anything |

💼 **Why this matters for consulting:** a client can use a third-party MCP server and still keep a **record of every action** the agent takes (useful for compliance: EU AI Act, GDPR), **without changing the server or the agent.**

---

## ✅ Final Summary

| Concept | In one line |
|---|---|
| **`await`** | "Wait here for the server's answer. Meanwhile, Python may run other ready tasks." |
| **Interceptor** | The receptionist between the agent and the tool, living in the MCP client. We used it to **log** every tool call; it can also **validate** and **change** them. The tool itself always runs on the server. |
